# Word-by-Word Caption Example

This notebook demonstrates how to create word-by-word captions using Auto-Caption.

## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

# Add the src directory to Python path if running from notebooks
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    src_path = notebook_dir.parent / 'src'
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

# Import required modules
from auto_caption.caption_generator import CaptionGenerator
from auto_caption.word_timing import WordTimingProcessor, WordAnimationStyle
from auto_caption.subtitle import ASSGenerator

## 2. Generate Basic Captions with Word Timestamps

In [ ]:
# Set your video path
video_path = "/path/to/your/video.mp4"  # Replace with your video file

# Create output directory
output_dir = "output/word_timing_demo"
os.makedirs(output_dir, exist_ok=True)

# Initialize caption generator
caption_gen = CaptionGenerator(
    model_name="base",
    verbose=True
)

# Generate captions with word-level timestamps
print("Generating captions with word timestamps...")
result = caption_gen.generate(
    video_path=video_path,
    word_timestamps=True,  # This enables word-level timestamps from Whisper
    temperature=0.0
)

print(f"\n✅ Generated {len(result['segments'])} segments")
print(f"🔤 Language detected: {result['language']}")

## 3. Process Segments for Word-by-Word Display

In [ ]:
# Initialize word timing processor
word_processor = WordTimingProcessor(
    animation_style=WordAnimationStyle.POP_IN,  # Choose animation style
    words_per_second=3.0,  # Average reading speed
    min_word_duration=0.15,  # Minimum time per word
    max_word_duration=0.8,  # Maximum time per word
)

# Process segments to create word-by-word timing
print("Processing segments for word-by-word display...")
word_segments = word_processor.process_segments(result['segments'])

print(f"\n✅ Processed {len(word_segments)} segments into word-by-word timing")

# Display first segment's word timing
if word_segments:
    first_segment = word_segments[0]
    print(f"\n📝 First segment has {len(first_segment.words)} words:")
    for i, word in enumerate(first_segment.words[:5]):  # Show first 5 words
        print(f"   {i+1}. '{word.word}' [{word.start_time:.2f}s - {word.end_time:.2f}s]")

## 4. Create Caption Data Structure

In [ ]:
# Create a copy of the result and add word segment data
word_result = result.copy()

# Add word segments to the result
word_result['word_segments'] = []
for segment in word_segments:
    segment_data = {
        'segment_index': segment.segment_index,
        'words': []
    }
    
    for word in segment.words:
        word_data = {
            'word': word.word,
            'start_time': word.start_time,
            'end_time': word.end_time,
            'duration': word.duration,
            'is_emphasized': word.is_emphasized
        }
        segment_data['words'].append(word_data)
    
    word_result['word_segments'].append(segment_data)

# Mark this as word-by-word caption data
word_result['word_by_word'] = True
word_result['word_animation'] = WordAnimationStyle.POP_IN.value

print("✅ Created word-by-word caption data structure")

## 5. Save Word-by-Word Captions

In [ ]:
# Save as JSON for inspection
json_path = os.path.join(output_dir, "word_captions.json")
caption_gen.save_output(word_result, json_path, "json")
print(f"✅ Saved JSON: {json_path}")

# Generate ASS subtitle file with word-by-word styling
ass_gen = ASSGenerator()
ass_path = os.path.join(output_dir, "word_captions.ass")
ass_gen.generate_ass_file(word_result, ass_path, "Word-by-Word Captions")
print(f"✅ Saved ASS: {ass_path}")

print(f"\n📁 All files saved to: {output_dir}")

## 6. Try Different Animation Styles

In [ ]:
# Available animation styles
animation_styles = [
    (WordAnimationStyle.TYPEWRITER, "Words appear sequentially"),
    (WordAnimationStyle.FADE_IN, "Words fade in smoothly"),
    (WordAnimationStyle.POP_IN, "Words pop in with scale effect"),
    (WordAnimationStyle.KARAOKE, "Karaoke-style highlighting"),
    (WordAnimationStyle.EMPHASIS, "Key words emphasized")
]

print("Available Animation Styles:")
print("=" * 50)

for style, description in animation_styles:
    print(f"\n{style.value.upper()}:")
    print(f"  Description: {description}")
    
    # Create processor with this style
    processor = WordTimingProcessor(
        animation_style=style,
        words_per_second=3.0
    )
    
    # Process segments
    styled_segments = processor.process_segments(result['segments'])
    
    # Create output for this style
    styled_result = result.copy()
    styled_result['word_segments'] = []
    
    for segment in styled_segments:
        segment_data = {
            'segment_index': segment.segment_index,
            'words': [{
                'word': w.word,
                'start_time': w.start_time,
                'end_time': w.end_time,
                'duration': w.duration,
                'is_emphasized': w.is_emphasized
            } for w in segment.words]
        }
        styled_result['word_segments'].append(segment_data)
    
    styled_result['word_by_word'] = True
    styled_result['word_animation'] = style.value
    
    # Save ASS file for this style
    style_ass_path = os.path.join(output_dir, f"captions_{style.value}.ass")
    ass_gen.generate_ass_file(styled_result, style_ass_path, f"{style.value.title()} Style")
    print(f"  Saved: {style_ass_path}")

## 7. Merge Captions with Video

In [ ]:
# Example command to merge captions with video using FFmpeg
print("To merge the captions with your video, use FFmpeg:")
print("\nFor word-by-word captions:")
print(f"ffmpeg -i {video_path} -vf \"subtitles={ass_path}\" output_with_words.mp4")
print("\nFor different styles:")
for style, _ in animation_styles[:3]:  # Show first 3 examples
    style_path = os.path.join(output_dir, f"captions_{style.value}.ass")
    print(f"ffmpeg -i {video_path} -vf \"subtitles={style_path}\" output_{style.value}.mp4")